# parsing.ms_office.markitdown.linux
> Recursively convert Microsoft Office files with MarkItDown on Linux.

The final cell reads `OFFICE_FILES_ROOT` from `PROJ_ROOT/.env`, writes a mirrored Markdown tree under `<OFFICE_FILES_ROOT>/.md`, extracts embedded base64 images into `img/` folders, and replaces the data URIs with relative image links.

In [ ]:
# |default_exp parsing.ms_office.markitdown.linux

In [ ]:
# | hide
from nbdev.showdoc import *
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

## Configure the recursive conversion pipeline

Install the notebook dependencies in the active Jupyter kernel if needed:

```bash
python -m pip install "markitdown[all]" python-dotenv
```

In [ ]:
# | export
import base64
import binascii
import os
import re
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv
from tqdm.auto import tqdm

In [ ]:
#| export
def _find_project_root(start: Path | str | None = None) -> Path:
    """Find PROJ_ROOT from the environment or a parent pyproject.toml."""
    configured_root = os.getenv("PROJ_ROOT")
    if configured_root:
        project_root = Path(configured_root).expanduser().resolve()
        if not project_root.is_dir():
            raise FileNotFoundError(f"PROJ_ROOT is not a directory: {project_root}")
        return project_root

    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(f"Could not find pyproject.toml above {current}")


PROJ_ROOT = _find_project_root()
ENV_FILE = PROJ_ROOT / ".env"
if not ENV_FILE.is_file():
    raise FileNotFoundError(f"Project environment file not found: {ENV_FILE}")
load_dotenv(ENV_FILE, override=False)


def get_office_files_root() -> Path:
    """Read and validate OFFICE_FILES_ROOT from PROJ_ROOT/.env."""
    configured_root = os.getenv("OFFICE_FILES_ROOT")
    if not configured_root:
        raise RuntimeError(f"OFFICE_FILES_ROOT is not configured in {ENV_FILE}")

    root = Path(configured_root).expanduser()
    if not root.is_absolute():
        root = PROJ_ROOT / root
    root = root.resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"OFFICE_FILES_ROOT is not a directory: {root}")
    return root

In [ ]:
#| export
OFFICE_EXTENSIONS = frozenset({
    ".doc", ".docx", ".odt",
    ".ppt", ".pptx", ".odp",
    ".xls", ".xlsx", ".xlsm", ".xlsb", ".ods",
    ".csv", ".tsv",
})

DATA_IMAGE_RE = re.compile(
    r"!\[(?P<alt>[^\]]*)\]\(\s*data:image/"
    r"(?P<mime>[-\w.+]+);base64,(?P<data>[A-Za-z0-9+/=\s]+?)\s*\)",
    flags=re.IGNORECASE,
)

In [ ]:
#| export
def convert_office_to_md(
    root_folder: Path | str,
    output_root: Path | str | None = None,
    *,
    overwrite: bool = False,
) -> dict[str, list]:
    """
    Recursively convert supported Office files to Markdown with MarkItDown.

    The output mirrors the input tree under ``output_root``. Each document
    gets its own directory: ``<relative parent>/<stem>/<stem>.md``.
    """
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Office root is not a directory: {root}")

    md_root = Path(output_root).expanduser().resolve() if output_root else root / ".md"
    md_root.mkdir(parents=True, exist_ok=True)
    report = {"converted": [], "skipped": [], "failed": []}
    source_files = sorted(
        path for path in root.rglob("*")
        if path.is_file()
        and path.suffix.lower() in OFFICE_EXTENSIONS
        and not path.name.startswith("~$")
    )

    for source in tqdm(
        source_files,
        desc="Converting Office files",
        unit="file",
        dynamic_ncols=True,
    ):
        relative_source = source.relative_to(root)
        markdown_file = (
            md_root / relative_source.parent / source.stem / f"{source.stem}.md"
        )
        if markdown_file.exists() and not overwrite:
            report["skipped"].append(markdown_file)
            print(f"Skipped existing: {markdown_file}")
            continue

        markdown_file.parent.mkdir(parents=True, exist_ok=True)
        command = [
            sys.executable,
            "-m",
            "markitdown",
            str(source),
            "-o",
            str(markdown_file),
            "--keep-data-uris",
        ]
        try:
            subprocess.run(command, check=True)
        except (OSError, subprocess.CalledProcessError) as error:
            report["failed"].append((source, error))
            print(f"Failed: {source}: {error}")
            continue

        report["converted"].append(markdown_file)
        print(f"Converted: {source} -> {markdown_file}")

    return report


In [ ]:
# MarkItDown must be available on PATH. Its --keep-data-uris option preserves
# embedded images so the extraction pass below can write them as image files.

In [ ]:
#| hide
def _extract_base64_images_legacy(markdown_file_path, image_output_folder="."):
    """
    Extracts base64 embedded images from a Markdown file, saves them to a folder,
    and replaces the base64 strings with relative paths to the new image files.

    Args:
        markdown_file_path (str): Path to the input Markdown file.
        image_output_folder (str): Name of the folder to save extracted images.
                                   This folder will be created relative to the
                                   Markdown file's directory if it doesn't exist.
    """
    if not os.path.exists(markdown_file_path):
        print(f"Error: Markdown file not found at {markdown_file_path}")
        return
    # markdown_file_stem = markdown_file_path.stem
    markdown_dir = os.path.dirname(os.path.abspath(markdown_file_path))
    full_image_output_path = os.path.join(markdown_dir, image_output_folder)

    if not os.path.exists(full_image_output_path):
        os.makedirs(full_image_output_path)
        # print(f"Created image output folder: {full_image_output_path}")

    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Regex to find base64 encoded images in Markdown
    # Pattern: ![alt text](data:image/png;base64,BASE64_STRING)
    # Groups:
    # 1: Alt text
    # 2: Image format (e.g., png, jpeg)
    # 3: Base64 data string
    # We also capture the full match (group 0) to replace it
    regex_img_quote = r"!\[(.*?)\]\(data:image/(.+?);base64,([A-Za-z0-9+/=\s]+)\)"
    regex_illegal_file_name = r'[^a-zA-Z0-9_\-\.]+'  # Legal characters for filenames

    new_content = content
    images_extracted_count = 0

    # We need to iterate carefully as string replacements change string length
    # Finding all matches first and then replacing is safer, but can be tricky
    # if matches overlap (not typical for this pattern).
    # A simpler approach for non-overlapping, distinct matches is to iterate
    # and replace. For more complex scenarios, one might work on a list of lines
    # or use re.sub with a function.

    # Using re.finditer to get match objects for more control
    for i, match in enumerate(re.finditer(regex_img_quote, content)):
        full_match_str = match.group(0)
        alt_text = match.group(1)
        # Normalize alt text to a legal filename
        alt_text = re.sub(regex_illegal_file_name, '_', alt_text)  # Replace illegal characters with '_'
        alt_text = alt_text.strip()  # Remove leading/trailing whitespace
        alt_text = alt_text[:50] if len(alt_text) > 50 else alt_text  # Limit length to 50 characters
        alt_text = 'img' if not alt_text else alt_text # If alt text is empty, use a default name

        image_format = match.group(2).lower() # e.g., png, jpeg
        image_format = re.sub(r'x-([a-zA-Z])', r'\1', image_format) # Normalize format (e.g., x-wmf/x-emf to wmf/emf)
        base64_data = match.group(3)

        # Clean up base64 data (remove potential whitespace)
        base64_data_cleaned = "".join(base64_data.split())
        # Fix missing padding
        missing_padding = len(base64_data_cleaned) % 4
        if missing_padding != 0:
            base64_data_cleaned += '=' * (4 - missing_padding)
        try:
            image_data = base64.b64decode(base64_data_cleaned)
        except base64.binascii.Error as e:
            print(f"Warning: Could not decode base64 string for an image (alt: {alt_text}). Error: {e}")
            continue # Skip this image

        # Generate a unique filename
        # Using a counter is simple, could use uuid for more robustness
        image_filename = f"{alt_text}_{images_extracted_count}.{image_format}"
        image_filepath = os.path.join(full_image_output_path, image_filename)

        # Save the image
        with open(image_filepath, 'wb') as img_file:
            img_file.write(image_data)
        # print(f"Extracted and saved: {image_filepath}")
        if image_format == 'wmf': # in case of wmf, we need to convert it to svg with soffice
            svg_file = Path(image_filepath).with_suffix('.svg') 
            subprocess.run([
                'soffice', '--headless', '--convert-to', 'svg',
                str(image_filepath),
                '--outdir', str(full_image_output_path),
            ], check=True)
            # if svg_file.exists():
            #     Path(image_filepath).unlink()  # Remove the original WMF file
                # image_filename = svg_file.name  # Update filename to the new SVG file

            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'soffice', '--headless', '--convert-to', 'png',
                str(image_filepath),
                '--outdir', str(full_image_output_path),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        elif image_format == 'emf': # in case of wmf, we need to convert it to svg with soffice
            svg_file = Path(image_filepath).with_suffix('.svg') 
            subprocess.run([
                'soffice', '--headless', '--convert-to', 'svg',
                str(image_filepath),
                '--outdir', str(full_image_output_path),
            ], check=True)
            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'soffice', '--headless', '--convert-to', 'png',
                str(image_filepath),
                '--outdir', str(full_image_output_path),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        elif image_format == 'gif': # in case of gif, we need to convert it to png with imagemagick convert
            png_file = Path(image_filepath).with_suffix('.png')
            subprocess.run([
                'convert', str(image_filepath),
                str(png_file),
            ], check=True)
            if png_file.exists():
                # Path(image_filepath).unlink()  # Remove the original WMF file
                image_filename = png_file.name  # Update filename to the new SVG file
        # Create the new Markdown image link (relative path)
        # The path in Markdown should be relative to the Markdown file itself
        relative_image_path = os.path.join(image_output_folder, image_filename)
        # Ensure forward slashes for Markdown paths, even on Windows
        relative_image_path_markdown = relative_image_path.replace(os.sep, '/')
        new_image_md_link = f"![{alt_text}]({relative_image_path_markdown})"

        # Replace the original base64 string with the new link in the `new_content`
        # Only replace the first occurrence of this specific full_match_str in case of duplicates
        # (though each match from finditer is unique in its position)
        new_content = new_content.replace(full_match_str, new_image_md_link, 1)
        images_extracted_count += 1

    if images_extracted_count > 0:
        # Save the modified Markdown content
        # You might want to save to a new file, e.g., original_filename_modified.md
        # For this example, I'll overwrite the original. Be careful!
        # Consider backing up your original file first.
        output_markdown_file_path = markdown_file_path # Overwrite
        # output_markdown_file_path = os.path.splitext(markdown_file_path)[0] + "_modified.md" # New file

        with open(output_markdown_file_path, 'w', encoding='utf-8') as f:
            f.write(new_content)
        print(f"Modified Markdown saved to: {output_markdown_file_path}, processed {images_extracted_count} image(s).")
    else:
        print("No base64 embedded images found in the Markdown file.")


In [ ]:
#| export
_IMAGE_SUFFIXES = {
    "jpeg": ".jpg",
    "jpg": ".jpg",
    "png": ".png",
    "gif": ".gif",
    "webp": ".webp",
    "bmp": ".bmp",
    "svg+xml": ".svg",
    "x-wmf": ".wmf",
    "x-emf": ".emf",
    "vnd.microsoft.icon": ".ico",
}


def _image_suffix(mime_subtype: str) -> str:
    normalized = mime_subtype.lower()
    if normalized in _IMAGE_SUFFIXES:
        return _IMAGE_SUFFIXES[normalized]
    safe_subtype = re.sub(r"[^a-z0-9]+", "_", normalized).strip("_")
    return f".{safe_subtype or 'bin'}"


def extract_base64_images(
    markdown_file_path: Path | str,
    image_output_folder: Path | str = "img",
) -> int:
    """Extract Markdown data-URI images and replace them with relative links."""
    markdown_file = Path(markdown_file_path).expanduser().resolve()
    if not markdown_file.is_file():
        raise FileNotFoundError(f"Markdown file not found: {markdown_file}")

    relative_image_dir = Path(image_output_folder)
    if relative_image_dir.is_absolute() or ".." in relative_image_dir.parts:
        raise ValueError("image_output_folder must stay inside the Markdown directory")

    image_dir = markdown_file.parent / relative_image_dir
    content = markdown_file.read_text(encoding="utf-8")
    extracted_count = 0

    def replace_data_uri(match: re.Match) -> str:
        nonlocal extracted_count
        alt_text = match.group("alt")
        encoded = "".join(match.group("data").split())
        encoded += "=" * (-len(encoded) % 4)
        try:
            image_data = base64.b64decode(encoded, validate=True)
        except (ValueError, binascii.Error) as error:
            print(f"Invalid base64 image in {markdown_file}: {error}")
            return match.group(0)

        extracted_count += 1
        safe_alt = re.sub(r"[^\w.-]+", "_", alt_text, flags=re.UNICODE).strip("._")
        safe_alt = (safe_alt[:50] or "image")
        image_name = (
            f"{extracted_count:04d}_{safe_alt}{_image_suffix(match.group('mime'))}"
        )
        image_dir.mkdir(parents=True, exist_ok=True)
        (image_dir / image_name).write_bytes(image_data)
        image_link = (relative_image_dir / image_name).as_posix()
        return f"![{alt_text}]({image_link})"

    rewritten = DATA_IMAGE_RE.sub(replace_data_uri, content)
    if rewritten != content:
        markdown_file.write_text(rewritten, encoding="utf-8")
    return extracted_count

# --- How to use it ---
if False:
    # Create a dummy Markdown file for testing
    dummy_md_content = """
# My Document

This is some text.

Here is an image: ![A red dot](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAUAAAAFCAYAAACNbyblAAAAHElEQVQI12P4//8/w38GIAXDIBKE0DHxgljNBAAO9TXL0Y4OHwAAAABJRU5ErkJggg==)

Some more text.

And another one: ![A blue square](data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAYABgAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAxNDQ0Hyc5PTgyPC4zNDL/2wBDAQkJCQwLDBgNDRgyIRwhMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjL/wAARCAAFAAUDASIAAhEBAxEB/8QAFQABAQAAAAAAAAAAAAAAAAAAAAb/xAAgEAACAQMEAwAAAAAAAAAAAAABAgADBBESBSFBUSKR/8QAFAEBAAAAAAAAAAAAAAAAAAAAAP/EABQRAQAAAAAAAAAAAAAAAAAAAAD/2gAMAwEAAhEDEQA/AIXVAvLBYy2PSkKOwz0A9YgA//Z)

This one is tricky with potential newlines in base64:
![With Newlines](data:image/gif;base64,R0lGODlhAQABAIAAAP///wAAACH5BAEAAAAALAAAAAABAAEAAAICRAEAOw==
)

End of document.
    """
    test_md_file = "test_document.md"
    with open(test_md_file, "w", encoding="utf-8") as f:
        f.write(dummy_md_content)
    print(f"Created dummy Markdown file: {test_md_file}")

    # Specify the path to your Markdown file
    markdown_file = test_md_file  # Or "your_actual_file.md"
    # Specify the folder (relative to the MD file) where images will be saved
    image_folder = "md_images"

    extract_base64_images(markdown_file, image_folder)

    # --- Optional: Clean up dummy files and folder after testing ---
    print("\nCleaning up dummy files...")
    if os.path.exists(os.path.join(os.path.dirname(test_md_file), image_folder)):
        for img_file in os.listdir(os.path.join(os.path.dirname(test_md_file), image_folder)):
            os.remove(os.path.join(os.path.dirname(test_md_file), image_folder, img_file))
        os.rmdir(os.path.join(os.path.dirname(test_md_file), image_folder))
    if os.path.exists(test_md_file):
        os.remove(test_md_file)
    print("Cleanup complete.")

In [ ]:
#| export
def extract_base64_from_md(
    root_folder: Path | str,
    image_output_folder: Path | str = "img",
) -> dict[str, object]:
    """
    Recursively extract data-URI images from every Markdown file under root.
    """
    root = Path(root_folder).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Markdown root is not a directory: {root}")

    report: dict[str, object] = {"processed": [], "image_count": 0, "failed": []}
    markdown_files = sorted(
        path for path in root.rglob("*.md") if path.is_file()
    )
    for markdown_file in tqdm(
        markdown_files,
        desc="Extracting base64 images",
        unit="file",
        dynamic_ncols=True,
    ):
        try:
            image_count = extract_base64_images(markdown_file, image_output_folder)
        except (OSError, ValueError) as error:
            report["failed"].append((markdown_file, error))
            print(f"Failed to extract images from {markdown_file}: {error}")
            continue

        report["processed"].append(markdown_file)
        report["image_count"] += image_count
        print(f"Extracted {image_count} image(s): {markdown_file}")

    return report

In [ ]:
# The final cell runs extraction after conversion; no hard-coded paths are used.

In [ ]:
#| export
def process_office_files(
    root_folder: Path | str,
    *,
    output_root: Path | str | None = None,
    overwrite: bool = False,
) -> dict[str, object]:
    """
    Convert all Office files recursively, then extract embedded images.
    """
    root = Path(root_folder).expanduser().resolve()
    md_root = Path(output_root).expanduser().resolve() if output_root else root / ".md"
    conversion = convert_office_to_md(root, md_root, overwrite=overwrite)
    extraction = extract_base64_from_md(md_root)

    print(
        "Finished: "
        f"{len(conversion['converted'])} converted, "
        f"{len(conversion['skipped'])} skipped, "
        f"{len(conversion['failed'])} conversion failure(s), "
        f"{extraction['image_count']} image(s) extracted, "
        f"{len(extraction['failed'])} extraction failure(s)."
    )
    return {
        "root": root,
        "output_root": md_root,
        "conversion": conversion,
        "extraction": extraction,
    }

In [ ]:
# Run the complete recursive pipeline configured by PROJ_ROOT/.env.
OFFICE_FILES_ROOT = get_office_files_root()
processing_report = process_office_files(OFFICE_FILES_ROOT, overwrite=False)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()

<div>
<link rel="stylesheet" href="https://gradio.s3-us-west-2.amazonaws.com/2.6.5/static/bundle.css">
<div id="target"></div>
<script src="https://gradio.s3-us-west-2.amazonaws.com/2.6.5/static/bundle.js"></script>
<script>
launchGradioFromSpaces("abidlabs/question-answering", "#target")
</script>
</div>